In [12]:
import re
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    pipeline
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

In [31]:
DATA_PATH = "reviews_mobile_jkn_20260607_193208.csv"  # ganti sesuai path dataset kamu

df = pd.read_csv(DATA_PATH)

print(df.head())
print(df.columns)
print(df.shape)

                userName                                            content  \
0                 Rifa'i                 tiap update harus login, nyusahin.   
1              joe Talim  lebih di tingkatkan lagi.. pake notif kyaknya ...   
2                   zawa  mempersulit bukan mempermudah kode otp aja ga ...   
3             Juli Yanah  saya kasih bintangnya full, semoga di mudah de...   
4  M Jevon Juli Ariyanto                                           membantu   

   score  
0      1  
1      4  
2      1  
3      5  
4      5  
Index(['userName', 'content', 'score'], dtype='str')
(15000, 3)


In [32]:
TARGET_COL = "content"

# pastikan kolom comment ada
assert TARGET_COL in df.columns, f"Kolom {TARGET_COL} tidak ditemukan."

# hapus missing value pada comment
df = df.dropna(subset=[TARGET_COL]).copy()

# ubah ke string
df[TARGET_COL] = df[TARGET_COL].astype(str)

# hapus comment kosong
df = df[df[TARGET_COL].str.strip() != ""].copy()

# hapus duplikat comment
df = df.drop_duplicates(subset=[TARGET_COL]).reset_index(drop=True)

print(df.shape)
df.head()

(10839, 3)


,userName,content,score
0,Rifa'i,"tiap update harus login, nyusahin.",1
1,joe Talim,lebih di tingkatkan lagi.. pake notif kyaknya ...,4
2,zawa,mempersulit bukan mempermudah kode otp aja ga ...,1
3,Juli Yanah,"saya kasih bintangnya full, semoga di mudah de...",5
4,M Jevon Juli Ariyanto,membantu,5


In [33]:
import re

def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticon
        "\U0001F300-\U0001F5FF"  # symbol & pictograph
        "\U0001F680-\U0001F6FF"  # transport & map
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U00002600-\U000026FF"  # misc symbols
        "\U0001F900-\U0001F9FF"  # supplemental symbols
        "\U0001FA70-\U0001FAFF"  # extended symbols
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(" ", text)

def clean_text(text):
    text = str(text).lower()
    
    # hapus URL
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # ubah mention jadi user
    text = re.sub(r"@\w+", " user ", text)
    
    # hapus hashtag simbolnya saja
    text = re.sub(r"#", "", text)
    
    # hapus emoji/emote
    text = remove_emoji(text)
    
    # hapus karakter aneh, tapi tetap simpan huruf, angka, spasi, dan tanda baca dasar
    text = re.sub(r"[^a-zA-Z0-9À-ÿ\u00C0-\u024F\s.,!?]", " ", text)
    
    # rapikan spasi
    text = re.sub(r"\s+", " ", text).strip()
    

    
    return text

In [34]:
df["clean_comment"] = df["content"].apply(clean_text)

df[["content", "clean_comment"]].head(20)

,content,clean_comment
0,"tiap update harus login, nyusahin.","tiap update harus login, nyusahin."
1,lebih di tingkatkan lagi.. pake notif kyaknya ...,lebih di tingkatkan lagi.. pake notif kyaknya ...
2,mempersulit bukan mempermudah kode otp aja ga ...,mempersulit bukan mempermudah kode otp aja ga ...
3,"saya kasih bintangnya full, semoga di mudah de...","saya kasih bintangnya full, semoga di mudah de..."
4,membantu,membantu
5,"APLIKASI LOLAAAAA BGTTTT, WIFI LANCAR, KUOTA M...","aplikasi lolaaaaa bgtttt, wifi lancar, kuota m..."
6,bagus,bagus
7,susah bgt pke apl ini. kadang² keluar sendiri ...,susah bgt pke apl ini. kadang keluar sendiri d...
8,perbaiki kinerjanya pada saat foto wajah,perbaiki kinerjanya pada saat foto wajah
9,nik yg dimasukkan sdh benar tapi apk nya bilan...,nik yg dimasukkan sdh benar tapi apk nya bilan...


In [35]:
ROBERTA_LABEL_MODEL = "w11wo/indonesian-roberta-base-sentiment-classifier"

labeler = pipeline(
    task="sentiment-analysis",
    model=ROBERTA_LABEL_MODEL,
    tokenizer=ROBERTA_LABEL_MODEL,
    device="cuda",
    truncation=True,
    max_length=128
)

texts = df["clean_comment"].tolist()

preds = labeler(
    texts,
    batch_size=64,
    truncation=True,
    max_length=128
)

df["pseudo_label_raw"] = [p["label"] for p in preds]
df["pseudo_label_score"] = [p["score"] for p in preds]

df[["clean_comment", "pseudo_label_raw", "pseudo_label_score"]].head()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9603.20it/s]


,clean_comment,pseudo_label_raw,pseudo_label_score
0,"tiap update harus login, nyusahin.",negative,0.995943
1,lebih di tingkatkan lagi.. pake notif kyaknya ...,positive,0.831265
2,mempersulit bukan mempermudah kode otp aja ga ...,negative,0.998148
3,"saya kasih bintangnya full, semoga di mudah de...",positive,0.936029
4,membantu,negative,0.487502


In [36]:
def normalize_label(label):
    label = str(label).lower()
    
    if "negative" in label or "negatif" in label or label in ["label_0", "0"]:
        return "negative"
    elif "neutral" in label or "netral" in label or label in ["label_1", "1"]:
        return "neutral"
    elif "positive" in label or "positif" in label or label in ["label_2", "2"]:
        return "positive"
    else:
        return label

df["label"] = df["pseudo_label_raw"].apply(normalize_label)

print(df["label"].value_counts())
df[["clean_comment", "pseudo_label_raw", "label", "pseudo_label_score"]].head()

label
negative    6361
positive    3301
neutral     1177
Name: count, dtype: int64


,clean_comment,pseudo_label_raw,label,pseudo_label_score
0,"tiap update harus login, nyusahin.",negative,negative,0.995943
1,lebih di tingkatkan lagi.. pake notif kyaknya ...,positive,positive,0.831265
2,mempersulit bukan mempermudah kode otp aja ga ...,negative,negative,0.998148
3,"saya kasih bintangnya full, semoga di mudah de...",positive,positive,0.936029
4,membantu,negative,negative,0.487502


In [61]:
df.to_csv("jkn_reviews_labeled.csv", index=False)

In [37]:
CONFIDENCE_THRESHOLD = 0.70

df_trainable = df[df["pseudo_label_score"] >= CONFIDENCE_THRESHOLD].copy()
df_trainable = df_trainable.reset_index(drop=True)

print("Data awal:", df.shape)
print("Data setelah filter confidence:", df_trainable.shape)
print(df_trainable["label"].value_counts())

Data awal: (10839, 7)
Data setelah filter confidence: (9804, 7)
label
negative    5989
positive    2975
neutral      840
Name: count, dtype: int64


In [38]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {v: k for k, v in label2id.items()}

df_trainable = df_trainable[df_trainable["label"].isin(label2id.keys())].copy()
df_trainable["labels"] = df_trainable["label"].map(label2id)

print(df_trainable["labels"].value_counts())
df_trainable.head(200)

labels
0    5989
2    2975
1     840
Name: count, dtype: int64


,userName,content,score,clean_comment,pseudo_label_raw,pseudo_label_score,label,labels
0,Rifa'i,"tiap update harus login, nyusahin.",1,"tiap update harus login, nyusahin.",negative,0.995943,negative,0
1,joe Talim,lebih di tingkatkan lagi.. pake notif kyaknya ...,4,lebih di tingkatkan lagi.. pake notif kyaknya ...,positive,0.831265,positive,2
2,zawa,mempersulit bukan mempermudah kode otp aja ga ...,1,mempersulit bukan mempermudah kode otp aja ga ...,negative,0.998148,negative,0
3,Juli Yanah,"saya kasih bintangnya full, semoga di mudah de...",5,"saya kasih bintangnya full, semoga di mudah de...",positive,0.936029,positive,2
4,sadiman 11 juli,"APLIKASI LOLAAAAA BGTTTT, WIFI LANCAR, KUOTA M...",1,"aplikasi lolaaaaa bgtttt, wifi lancar, kuota m...",positive,0.765584,positive,2
...,...,...,...,...,...,...,...,...
195,siti fatimah,apk gk jelas,1,apk gk jelas,negative,0.997961,negative,0
196,Syahdan Jambi,sangat membantu tentang infonya,5,sangat membantu tentang infonya,positive,0.987668,positive,2
197,agung prasetyo,sangat mudah,5,sangat mudah,positive,0.927767,positive,2
198,Zainul Anwar,tak kasih bintang 5 supaya bpjs aku dan istri ...,5,tak kasih bintang 5 supaya bpjs aku dan istri ...,neutral,0.716377,neutral,1


In [46]:
train_df, val_df = train_test_split(
    df_trainable,
    test_size=0.2,
    random_state=42,
    stratify=df_trainable["labels"]
)

print(train_df.shape)
print(val_df.shape)

(7843, 8)
(1961, 8)


In [47]:
MODEL_NAME = "indolem/indobertweet-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df[["clean_comment", "labels"]])
val_dataset = Dataset.from_pandas(val_df[["clean_comment", "labels"]])

def tokenize_function(batch):
    return tokenizer(
        batch["clean_comment"],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["clean_comment"])
val_dataset = val_dataset.remove_columns(["clean_comment"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

Map: 100%|██████████| 1961/1961 [00:00<00:00, 19874.23 examples/s]
[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33254.97it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPEC

In [51]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )
    
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

In [52]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none"
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [53]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.072817,0.289998,0.939827,0.900779,0.846120,0.869046
2,0.104148,0.241715,0.949516,0.911764,0.869297,0.887755
3,0.049393,0.330131,0.947986,0.919158,0.857071,0.881542
4,0.022398,0.357825,0.946966,0.914946,0.853103,0.877208


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.56it/s]


TrainOutput(global_step=1964, training_loss=0.06323147718955871, metrics={'train_runtime': 228.8801, 'train_samples_per_second': 171.334, 'train_steps_per_second': 10.726, 'total_flos': 980838515310948.0, 'train_loss': 0.06323147718955871, 'epoch': 4.0})

In [54]:
eval_result = trainer.evaluate()
eval_result

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
0.022398,0.241715,4,0.949516,0.911764,0.869297,0.887755


{'eval_loss': 0.2417154610157013,
 'eval_accuracy': 0.9495155532891382,
 'eval_macro_precision': 0.9117642983576676,
 'eval_macro_recall': 0.8692969842360984,
 'eval_macro_f1': 0.8877549246167847}

In [55]:
pred_output = trainer.predict(val_dataset)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))],
    zero_division=0
))

print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

    negative       0.96      0.98      0.97      1198
     neutral       0.82      0.66      0.73       168
    positive       0.96      0.96      0.96       595

    accuracy                           0.95      1961
   macro avg       0.91      0.87      0.89      1961
weighted avg       0.95      0.95      0.95      1961

[[1177   15    6]
 [  39  111   18]
 [  11   10  574]]


In [56]:
SAVE_DIR = "./final-indobertweet-comment-sentiment"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model saved to:", SAVE_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s]

Model saved to: ./final-indobertweet-comment-sentiment


In [57]:
sentiment_pipe = pipeline(
    task="text-classification",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=128
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13445.00it/s]


In [59]:
new_comments = [
    "aplikasinya sering error dan lambat",
    "biasa aja sih menurutku",
    "bagus banget, sangat membantu"
]

new_df = pd.DataFrame({"comment": new_comments})
new_df["clean_comment"] = new_df["comment"].apply(clean_text)

predictions = sentiment_pipe(
    new_df["clean_comment"].tolist(),
    batch_size=16,
    truncation=True,
    max_length=128
)

new_df["pred_label"] = [p["label"] for p in predictions]
new_df["pred_score"] = [p["score"] for p in predictions]

new_df

,comment,clean_comment,pred_label,pred_score
0,aplikasinya sering error dan lambat,aplikasinya sering error dan lambat,negative,0.999811
1,biasa aja sih menurutku,biasa aja sih menurutku,negative,0.993104
2,"bagus banget, sangat membantu","bagus banget, sangat membantu",positive,0.999864
